# 10. Comparación estadística de los modelos

## 10.1. El problema de escala

El experimento produce 112 combinaciones. El número de comparaciones por pares posibles entre ellas es 112 × 111 / 2 = 6,216. Ejecutar esa cantidad de pruebas con un nivel de significancia de 0.05 y sin ningún control produciría alrededor de 311 resultados significativos solo por azar.

Por eso la comparación no puede reducirse a aplicar pruebas por pares. Se sigue la secuencia jerárquica que exige la guía:

1. **Prueba ómnibus (Friedman)** sobre los rangos de los modelos en los folds del bucle externo. Si no rechaza, no hay evidencia de que ningún modelo supere a los demás y el análisis se detiene.
2. **Post-hoc (Nemenyi) y diagrama de diferencia crítica**, que compara todos contra todos controlando el error familiar de forma conjunta.
3. **Comparaciones dirigidas (DeLong)** entre un subconjunto reducido y justificado de modelos, con corrección de Holm-Bonferroni.
4. **Tamaño del efecto (delta de Cliff)** en todos los casos, favorables o no.
5. **Intervalos de confianza por bootstrap BCa** para las métricas principales.

In [ ]:
from config import *

maestra = pd.read_csv(RESULTADOS / "tabla_maestra_clasificacion.csv")
completadas = maestra.query("estado == 'completada'").copy()
completadas["configuracion"] = (completadas["modelo"] + " | "
                                + completadas["balanceo"] + " | "
                                + completadas["optimizador"])

# Matriz de métricas por fold: filas = folds externos, columnas = modelos.
# Es la estructura que requieren Friedman y Nemenyi, que tratan los folds
# como bloques de medidas repetidas.
por_fold = pd.DataFrame({
    fila["configuracion"]: json.loads(fila["auc_pr_por_fold"])
    for _, fila in completadas.iterrows()
})
por_fold.index = [f"fold {i}" for i in range(1, len(por_fold) + 1)]

print(f"Configuraciones comparadas: {por_fold.shape[1]}")
print(f"Bloques (folds externos)  : {por_fold.shape[0]}")
print(f"Comparaciones por pares posibles: "
      f"{por_fold.shape[1] * (por_fold.shape[1] - 1) // 2:,}")

## 10.2. Etapa 1: prueba ómnibus de Friedman

La prueba de Friedman es el análogo no paramétrico del ANOVA de medidas repetidas. Convierte las métricas en rangos dentro de cada fold y contrasta la hipótesis nula de que los rangos promedio son iguales entre modelos. Es la prueba apropiada aquí por dos razones: no supone normalidad de las métricas, y trata cada fold como un bloque, lo que reconoce que las mediciones de todos los modelos en un mismo fold están correlacionadas por compartir los mismos datos.

Se aplica en dos niveles. Primero sobre los siete modelos base, usando su mejor configuración, que es la comparación de interés principal. Después sobre las 112 configuraciones completas.

In [ ]:
mejores_por_modelo = (completadas.loc[
    completadas.groupby("modelo")["auc_pr_media"].idxmax()])

por_fold_modelos = pd.DataFrame({
    fila["modelo"]: json.loads(fila["auc_pr_por_fold"])
    for _, fila in mejores_por_modelo.iterrows()
})

estadistico, p_valor = stats.friedmanchisquare(
    *[por_fold_modelos[c] for c in por_fold_modelos.columns])

friedman = pd.DataFrame([{
    "comparación": "7 modelos base (mejor configuración de cada uno)",
    "k (modelos)": por_fold_modelos.shape[1],
    "n (folds)": por_fold_modelos.shape[0],
    "chi² de Friedman": estadistico,
    "grados de libertad": por_fold_modelos.shape[1] - 1,
    "p-valor": p_valor,
    "conclusión": ("Se rechaza H0: hay diferencias" if p_valor < 0.05
                   else "No se rechaza H0"),
}])
guardar_resultado(friedman, "friedman_modelos")
friedman.round(4)

In [ ]:
# Rangos promedio: 1 es el mejor. Es el estadístico que ordena el diagrama
# de diferencia crítica.
rangos = por_fold_modelos.rank(axis=1, ascending=False)
rangos_promedio = rangos.mean().sort_values()

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.barh(rangos_promedio.index[::-1], rangos_promedio.values[::-1],
        color=PALETA[0])
ax.set_xlabel("Rango promedio en AUC-PR (menor es mejor)")
ax.set_title("Rangos promedio de los modelos entre folds externos")
plt.tight_layout()
plt.show()

rangos_promedio.round(3).to_frame("rango promedio")

Si la prueba de Friedman **no** rechaza la hipótesis nula, el análisis debe detenerse aquí y así hay que reportarlo: no existe evidencia estadística de que algún modelo supere a los demás, independientemente de las diferencias observadas en las métricas puntuales. Con cinco folds y siete modelos, el poder de la prueba es limitado, de modo que un resultado no significativo es perfectamente posible y es una conclusión legítima, no un fracaso del experimento.

Conviene además reportar el número de bloques: la prueba de Friedman con cinco bloques tiene poco poder, y una forma de aumentarlo sin sesgar nada es repetir la validación externa con varias semillas y tratar cada repetición como bloque adicional.

## 10.3. Etapa 2: post-hoc de Nemenyi y diagrama de diferencia crítica

Si Friedman rechaza, la prueba de Nemenyi compara todos los pares controlando el error familiar. Su resultado se resume en la **diferencia crítica**: dos modelos cuya distancia en rango promedio sea menor que ese valor son estadísticamente indistinguibles.

$$
CD = q_{\alpha} \sqrt{\frac{k(k+1)}{6n}}
$$

donde *k* es el número de modelos, *n* el número de bloques y *q* el valor crítico de la distribución del rango studentizado dividido por la raíz de dos.

In [ ]:
import scikit_posthocs as sp

nemenyi = sp.posthoc_nemenyi_friedman(por_fold_modelos)
guardar_resultado(nemenyi, "nemenyi_modelos")

fig, ax = plt.subplots(figsize=(7, 5.5))
mascara = np.triu(np.ones_like(nemenyi, dtype=bool))
sns.heatmap(nemenyi, mask=mascara, annot=True, fmt=".3f", cmap="RdYlGn_r",
            vmin=0, vmax=1, linewidths=0.5, square=True,
            cbar_kws={"label": "p-valor de Nemenyi"}, ax=ax)
ax.set_title("Prueba post-hoc de Nemenyi (p-valores por pares)")
plt.tight_layout()
plt.show()

In [ ]:
def diferencia_critica(k, n, alfa=0.05):
    """Diferencia crítica de Nemenyi para k modelos y n bloques.

    Usa la aproximación habitual con el valor crítico del rango
    studentizado dividido por la raíz de dos.

    Returns
    -------
    float
    """
    q_alfa = stats.studentized_range.ppf(1 - alfa, k, np.inf) / np.sqrt(2)
    return float(q_alfa * np.sqrt(k * (k + 1) / (6 * n)))


def diagrama_cd(rangos_promedio, cd, titulo):
    """Dibuja un diagrama de diferencia crítica.

    Ubica cada modelo según su rango promedio y conecta con una línea
    horizontal los grupos cuya diferencia no alcanza la diferencia crítica.
    """
    ordenados = rangos_promedio.sort_values()
    nombres, valores = list(ordenados.index), ordenados.to_numpy()

    fig, ax = plt.subplots(figsize=(9, 2.2 + 0.32 * len(nombres)))
    ax.plot(valores, np.zeros_like(valores), "o", color=PALETA[0],
            markersize=8)
    for i, (nombre, valor) in enumerate(zip(nombres, valores)):
        altura = -0.12 * (i + 1)
        ax.plot([valor, valor], [0, altura], color=GRIS, lw=0.8)
        ax.text(valor, altura - 0.03, f"{nombre} ({valor:.2f})",
                ha="center", va="top", fontsize=9)

    # Grupos de modelos no distinguibles entre sí.
    nivel = 0.10
    for i in range(len(valores)):
        compatibles = [j for j in range(i, len(valores))
                       if valores[j] - valores[i] <= cd]
        if len(compatibles) > 1:
            fin = compatibles[-1]
            ax.plot([valores[i] - 0.02, valores[fin] + 0.02],
                    [nivel, nivel], color=PALETA[1], lw=3,
                    solid_capstyle="butt")
            nivel += 0.08

    ax.annotate("", xy=(valores.min(), nivel + 0.08),
                xytext=(valores.min() + cd, nivel + 0.08),
                arrowprops={"arrowstyle": "|-|", "lw": 1.2})
    ax.text(valores.min() + cd / 2, nivel + 0.12, f"CD = {cd:.2f}",
            ha="center", fontsize=9)
    ax.set_xlabel("Rango promedio (menor es mejor)")
    ax.set_ylim(-0.12 * (len(nombres) + 2), nivel + 0.22)
    ax.set_yticks([])
    ax.set_title(titulo)
    for lado in ["left", "right", "top"]:
        ax.spines[lado].set_visible(False)
    plt.tight_layout()
    plt.show()


cd = diferencia_critica(por_fold_modelos.shape[1], por_fold_modelos.shape[0])
print(f"Diferencia crítica (alfa = 0.05, k = {por_fold_modelos.shape[1]}, "
      f"n = {por_fold_modelos.shape[0]}): {cd:.3f}")
diagrama_cd(rangos_promedio, cd,
            "Diagrama de diferencia crítica — modelos base (AUC-PR)")

El diagrama sustituye a una tabla de miles de comparaciones: los modelos unidos por una barra horizontal no se distinguen estadísticamente entre sí.

Con solo cinco folds, la diferencia crítica es grande, y es muy posible que todos los modelos queden conectados. Esa conclusión es informativa y hay que presentarla como tal: el experimento no tiene poder suficiente para separar los modelos con este número de bloques, lo que refuerza lo que el capítulo 4 anticipaba sobre la dificultad intrínseca del problema.

## 10.4. Comparaciones dirigidas: prueba de DeLong

DeLong contrasta la igualdad de dos AUC calculadas sobre las **mismas observaciones**, lo que la hace apropiada aquí: los tres finalistas se evaluaron sobre el mismo conjunto de prueba, así que sus errores están correlacionados y una prueba que ignore esa correlación sería demasiado conservadora.

Se reserva para el subconjunto reducido de los tres finalistas del capítulo 9, tal como exige la guía, y sus valores p se ajustan por comparaciones múltiples.

In [ ]:
predicciones = pd.read_csv(RESULTADOS / "predicciones_prueba.csv")
y_real = predicciones["y_real"].to_numpy()
columnas_modelos = [c for c in predicciones.columns if c.startswith("p_")]


def _estructura_delong(y_real, probabilidades):
    """Componentes de la varianza del AUC según DeLong.

    Returns
    -------
    tuple
        AUC y las dos series de pseudovalores (V10 sobre positivos y V01
        sobre negativos) que definen su varianza.
    """
    positivos = probabilidades[y_real == 1]
    negativos = probabilidades[y_real == 0]
    m, n = len(positivos), len(negativos)

    # Kernel de comparación con empates ponderados a la mitad.
    comparaciones = (positivos[:, None] > negativos[None, :]).astype(float)
    comparaciones += 0.5 * (positivos[:, None] == negativos[None, :])

    auc = comparaciones.mean()
    v10 = comparaciones.mean(axis=1)   # un valor por positivo
    v01 = comparaciones.mean(axis=0)   # un valor por negativo
    return auc, v10, v01, m, n


def prueba_delong(y_real, probabilidades_a, probabilidades_b):
    """Prueba de DeLong para dos AUC correlacionados.

    Contrasta H0: AUC_A = AUC_B sobre las mismas observaciones, usando la
    representación del AUC como estadístico U y su matriz de covarianzas
    estimada a partir de los pseudovalores.

    Returns
    -------
    dict
        AUC de cada modelo, su diferencia, el estadístico z y el p-valor
        bilateral.
    """
    auc_a, v10_a, v01_a, m, n = _estructura_delong(y_real, probabilidades_a)
    auc_b, v10_b, v01_b, _, _ = _estructura_delong(y_real, probabilidades_b)

    s10 = np.cov(np.vstack([v10_a, v10_b]))
    s01 = np.cov(np.vstack([v01_a, v01_b]))
    covarianza = s10 / m + s01 / n

    diferencia = auc_a - auc_b
    varianza = covarianza[0, 0] + covarianza[1, 1] - 2 * covarianza[0, 1]
    z = diferencia / np.sqrt(varianza) if varianza > 0 else 0.0
    return {"auc_a": auc_a, "auc_b": auc_b, "diferencia": diferencia,
            "z": float(z), "p": float(2 * stats.norm.sf(abs(z)))}


def correccion_holm(valores_p):
    """Corrección de Holm-Bonferroni para comparaciones múltiples."""
    p = np.asarray(valores_p, dtype=float)
    m = len(p)
    orden = np.argsort(p)
    ajustados, maximo = np.empty(m), 0.0
    for i, indice in enumerate(orden):
        maximo = max(maximo, (m - i) * p[indice])
        ajustados[indice] = min(1.0, maximo)
    return ajustados

In [ ]:
from itertools import combinations

filas = []
for columna_a, columna_b in combinations(columnas_modelos, 2):
    resultado = prueba_delong(y_real,
                              predicciones[columna_a].to_numpy(),
                              predicciones[columna_b].to_numpy())
    filas.append({
        "modelo A": columna_a[2:], "modelo B": columna_b[2:],
        "AUC A": resultado["auc_a"], "AUC B": resultado["auc_b"],
        "diferencia": resultado["diferencia"],
        "z": resultado["z"], "p crudo": resultado["p"],
    })

delong = pd.DataFrame(filas)
delong["p (Holm)"] = correccion_holm(delong["p crudo"])
delong["p (Benjamini-Hochberg)"] = stats.false_discovery_control(
    delong["p crudo"], method="bh")
delong["significativa (Holm)"] = np.where(delong["p (Holm)"] < 0.05,
                                          "Sí", "No")
guardar_resultado(delong, "delong_finalistas")
delong.round(4)

Se reportan el valor p crudo y los dos ajustados, como pide la guía. La diferencia entre Holm y Benjamini-Hochberg es el criterio que controlan: Holm controla la probabilidad de cometer **al menos un** falso positivo en la familia de comparaciones, más estricto; Benjamini-Hochberg controla la **proporción esperada** de falsos positivos entre los rechazos, más permisivo y preferible cuando el objetivo es explorar y no confirmar.

## 10.5. Tamaño del efecto

La significancia estadística no implica relevancia práctica, y con múltiples folds el poder puede ser suficiente para detectar diferencias irrelevantes. Se reporta el tamaño del efecto en todos los casos.

La **delta de Cliff** es la medida recomendada junto a Friedman-Nemenyi por ser consistente con el uso de rangos: mide la probabilidad de que la métrica de un modelo supere a la del otro, menos la probabilidad inversa. Sus umbrales convencionales son 0.147 (pequeño), 0.330 (mediano) y 0.474 (grande).

In [ ]:
def delta_cliff(a, b):
    """Delta de Cliff entre dos muestras a partir del estadístico U."""
    u = stats.mannwhitneyu(a, b).statistic
    return 2 * u / (len(a) * len(b)) - 1


def magnitud_cliff(delta):
    """Clasifica la delta de Cliff según los umbrales convencionales."""
    d = abs(delta)
    if d < 0.147:
        return "Insignificante"
    if d < 0.330:
        return "Pequeño"
    if d < 0.474:
        return "Mediano"
    return "Grande"


filas = []
for modelo_a, modelo_b in combinations(por_fold_modelos.columns, 2):
    delta = delta_cliff(por_fold_modelos[modelo_a],
                        por_fold_modelos[modelo_b])
    filas.append({
        "modelo A": modelo_a, "modelo B": modelo_b,
        "AUC-PR medio A": por_fold_modelos[modelo_a].mean(),
        "AUC-PR medio B": por_fold_modelos[modelo_b].mean(),
        "delta de Cliff": delta,
        "magnitud": magnitud_cliff(delta),
    })

efectos = pd.DataFrame(filas).sort_values(
    "delta de Cliff", key=abs, ascending=False)
guardar_resultado(efectos, "tamanos_efecto_modelos")
tabla(efectos.round(4), filas=25)

## 10.6. Intervalos de confianza por bootstrap BCa

Reportar una métrica como un número puntual oculta su incertidumbre. Se construyen intervalos por bootstrap con corrección de sesgo y aceleración (BCa), que es el método apropiado cuando la distribución de la métrica es asimétrica, como suele ocurrir con el AUC-PR en problemas desbalanceados.

El remuestreo se hace **por paciente** y no por fila, coherente con la estructura de los datos: remuestrear filas trataría los encuentros de un mismo paciente como observaciones independientes y produciría intervalos demasiado estrechos.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

prueba = leer_tabla("diabetes_test")
identificadores = prueba[roles_variables()["identificador"]].to_numpy() \
    if "roles_variables" in dir() else None

roles = roles_variables()
identificadores = prueba[roles["identificador"]].to_numpy()


def bootstrap_bca(y_real, probabilidades, grupos, funcion_metrica,
                  n_repeticiones=2_000, confianza=0.95, semilla=RANDOM_STATE):
    """Intervalo de confianza BCa remuestreando grupos completos.

    Parameters
    ----------
    y_real, probabilidades : array-like
    grupos : array-like
        Identificador de paciente; el remuestreo toma pacientes completos.
    funcion_metrica : callable
        Recibe (y_real, probabilidades) y devuelve un escalar.
    n_repeticiones : int
    confianza : float
    semilla : int

    Returns
    -------
    dict
        Estimación puntual y límites del intervalo.
    """
    aleatorio = np.random.default_rng(semilla)
    y_real = np.asarray(y_real)
    probabilidades = np.asarray(probabilidades)
    unicos = np.unique(grupos)
    indices_por_grupo = {g: np.where(grupos == g)[0] for g in unicos}

    estimacion = funcion_metrica(y_real, probabilidades)

    replicas = []
    for _ in range(n_repeticiones):
        muestra = aleatorio.choice(unicos, size=len(unicos), replace=True)
        indices = np.concatenate([indices_por_grupo[g] for g in muestra])
        if len(np.unique(y_real[indices])) < 2:
            continue
        replicas.append(funcion_metrica(y_real[indices],
                                        probabilidades[indices]))
    replicas = np.asarray(replicas)

    # Corrección de sesgo: proporción de réplicas por debajo de la
    # estimación original, transformada a la escala normal.
    proporcion = np.mean(replicas < estimacion)
    proporcion = np.clip(proporcion, 1e-6, 1 - 1e-6)
    z0 = stats.norm.ppf(proporcion)

    # Aceleración por jackknife sobre los grupos.
    jackknife = []
    for g in unicos:
        indices = np.concatenate([idx for gg, idx in indices_por_grupo.items()
                                  if gg != g])
        if len(np.unique(y_real[indices])) < 2:
            continue
        jackknife.append(funcion_metrica(y_real[indices],
                                         probabilidades[indices]))
    jackknife = np.asarray(jackknife)
    desviaciones = jackknife.mean() - jackknife
    denominador = 6 * (np.sum(desviaciones ** 2) ** 1.5)
    aceleracion = (np.sum(desviaciones ** 3) / denominador
                   if denominador > 0 else 0.0)

    alfa = 1 - confianza
    inferior, superior = [], []
    for z_alfa in [stats.norm.ppf(alfa / 2), stats.norm.ppf(1 - alfa / 2)]:
        ajustado = z0 + (z0 + z_alfa) / (1 - aceleracion * (z0 + z_alfa))
        percentil = 100 * stats.norm.cdf(ajustado)
        (inferior if z_alfa < 0 else superior).append(
            np.percentile(replicas, np.clip(percentil, 0, 100)))

    return {"estimación": estimacion, "IC inferior": inferior[0],
            "IC superior": superior[0], "réplicas válidas": len(replicas)}

In [ ]:
filas = []
for columna in columnas_modelos:
    p = predicciones[columna].to_numpy()
    for nombre_metrica, funcion in [("AUC-ROC", roc_auc_score),
                                    ("AUC-PR", average_precision_score)]:
        resultado = bootstrap_bca(y_real, p, identificadores, funcion,
                                  n_repeticiones=1_000)
        filas.append({"modelo": columna[2:], "métrica": nombre_metrica,
                      **resultado})

intervalos = pd.DataFrame(filas)
guardar_resultado(intervalos, "intervalos_bca")
intervalos.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
subconjunto = intervalos.query("métrica == 'AUC-PR'").reset_index(drop=True)
posiciones = np.arange(len(subconjunto))
ax.errorbar(subconjunto["estimación"], posiciones,
            xerr=[subconjunto["estimación"] - subconjunto["IC inferior"],
                  subconjunto["IC superior"] - subconjunto["estimación"]],
            fmt="o", color=PALETA[1], capsize=4, markersize=7)
ax.set_yticks(posiciones)
ax.set_yticklabels(subconjunto["modelo"])
ax.axvline(y_real.mean(), ls="--", color=GRIS, lw=1, label="Prevalencia")
ax.set_xlabel("AUC-PR en el conjunto de prueba (IC 95 % BCa)")
ax.set_title("Intervalos de confianza de los finalistas")
ax.legend()
plt.tight_layout()
plt.show()

El solapamiento de intervalos es la lectura decisiva: si los intervalos de dos modelos se solapan, la diferencia entre ellos no es distinguible del ruido de muestreo, con independencia de cuál tenga la media más alta.

## 10.7. Síntesis del capítulo

| Etapa | Resultado a registrar |
|---|---|
| Friedman sobre los 7 modelos base | Estadístico, grados de libertad, p-valor y si se rechaza la hipótesis nula |
| Rangos promedio | Orden de los modelos y distancia entre ellos |
| Nemenyi y diagrama CD | Diferencia crítica y qué grupos de modelos quedan conectados, es decir, son indistinguibles |
| DeLong entre finalistas | Diferencias de AUC con p crudo, p de Holm y p de Benjamini-Hochberg |
| Delta de Cliff | Magnitud de cada comparación, reportada también cuando es insignificante |
| Intervalos BCa | IC del 95 % de AUC-ROC y AUC-PR de los finalistas, con el remuestreo hecho por paciente |

### 10.7.1. Cómo redactar la conclusión

La conclusión honesta depende de lo que muestren las pruebas, y conviene tener preparadas las dos redacciones:

- **Si Friedman rechaza y el diagrama CD separa grupos**, se puede afirmar que un subconjunto de modelos supera a otro, y la delta de Cliff dice si esa superioridad es además relevante en magnitud.
- **Si Friedman no rechaza, o el diagrama conecta a todos los modelos**, la conclusión es que con cinco folds el experimento no distingue entre modelos. Eso no invalida el trabajo: significa que en este problema la elección del modelo importa menos que la calidad de los datos disponibles, lo cual es coherente con los tamaños de efecto insignificantes que el capítulo 4 encontró en todos los predictores. Presentar esa conclusión con sus pruebas es más valioso que forzar una diferencia con pruebas sin corrección.